# **Implementing Uninformed, Informed, and Local Search Algorithms**

# Objectives
By the end of this practical session, you will be able to:
*   Understand the fundamental differences and trade-offs between uninformed search algorithms (Depth-First Search vs. Uniform-Cost Search).
*   Understand the role of heuristics in informed search algorithms (Greedy Best-First Search vs. A* Search).
*   Grasp the concepts and applications of local search algorithms for optimisation (Simulated Annealing and Local Beam Search).
* Learn the basic principles and components of evolutionary algorithms (Genetic Algorithms).


# Material Needed
*   A computer with Python 3.x installed.
*   A code editor (e.g., VS Code, PyCharm) or an online Python environment (e.g., Google Colab, Repl.it).
*   No special Python libraries are needed beyond standard ones like heapq, random, and math, which are usually pre-installed.

# **Part 2-1**

**A. Uniform-Cost Search vs. Depth-First Search**

We will start with two fundamental graph search algorithms: Uniform-Cost Search (UCS) and Depth-First Search (DFS). UCS is all about finding the cheapest path, while DFS is about exploring deep into paths quickly.

In [ ]:
# --- Depth-First Search (DFS) ---
# DFS dives as deep as possible down one path before trying another.
# It uses a stack to keep track of where it needs to backtrack to.
def dfs(graph, start, goal):
    """
    Performs Depth-First Search on a graph.
    Returns the path from start to goal if found, otherwise None.
    """
    stack = [(start, [start])] # Stack stores (current_node, current_path) (A,[A])
    visited = set() # Empty space

    print(f"\n--- Running DFS from {start} to {goal} ---")

    while stack:
        current_node, path = stack.pop() # Starting point = current_node = A, path = [A]
        print(f"  Exploring node: {current_node}, current path: {path}")

        if current_node == goal:
            print(f"  Goal {goal} reached!")
            return path

        if current_node not in visited:
            visited.add(current_node) # if 'A' not in Visited then add it
            # Add neighbors to stack in reverse order to explore in alphabetical order for consistency
            for neighbor in sorted(graph[current_node].keys(), reverse=True):
                if neighbor not in visited:
                    stack.append((neighbor, path + [neighbor]))
    print(f"  Goal {goal} not found.")
    return None

In [ ]:
# --- Uniform-Cost Search (UCS) ---
# UCS explores the path that has the lowest total cost from the start node so far.
# It guarantees finding the cheapest path.
# It uses a priority queue to always pick the lowest-cost path segment.
def ucs(graph, start, goal):
    """
    Performs Uniform-Cost Search on a graph.
    Returns the least-cost path from start to goal if found, otherwise None.
    """
    # Priority queue stores (cost, current_node, path)
    # ` min-heap property ensures lowest cost is always at the top
    priority_queue = [(0, start, [start])]
    visited_costs = {start: 0} # To keep track of the cheapest path found to a node

    print(f"\n--- Running UCS from {start} to {goal} ---")

    while priority_queue:
        cost, current_node, path = heapq.heappop(priority_queue)
        print(f"  Exploring node: {current_node} (Cost: {cost}), current path: {path}")

        # If we've already found a cheaper path to this node, we can skip this one
        if cost > visited_costs.get(current_node, float('inf')):
            continue

        if current_node == goal:
            print(f"  Goal {goal} reached with total cost: {cost}!")
            return path, cost

        for neighbor, edge_cost in graph[current_node].items():
            new_cost = cost + edge_cost
            # Only add to queue if this new path to neighbour is cheaper than any previous one
            if new_cost < visited_costs.get(neighbor, float('inf')):
                visited_costs[neighbor] = new_cost
                heapq.heappush(priority_queue, (new_cost, neighbor, path + [neighbor]))
    print(f"  Goal {goal} not found.")
    return None, None

In [ ]:
import heapq # For Uniform-Cost Search's priority queue

# Define a sample graph for demonstration
# Format: {node: {neighbor: cost}}
graph = {
    'A': {'B': 1, 'C': 5},
    'B': {'D': 3, 'E': 6},
    'C': {'F': 2, 'G': 4},
    'D': {'H': 1},
    'E': {'H': 7},
    'F': {'I': 3},
    'G': {'I': 2},
    'H': {'J': 1},
    'I': {'J': 4},
    'J': {}
}

print("--- Graph for DFS and UCS ---")
for node, neighbors in graph.items():
    print(f"{node}: {neighbors}")
print("-" * 30)

--- Graph for DFS and UCS ---
A: {'B': 1, 'C': 5}
B: {'D': 3, 'E': 6}
C: {'F': 2, 'G': 4}
D: {'H': 1}
E: {'H': 7}
F: {'I': 3}
G: {'I': 2}
H: {'J': 1}
I: {'J': 4}
J: {}
------------------------------


In [ ]:
# --- DFS Example Run ---
print("\n--- Testing DFS ---")
dfs_path = dfs(graph, 'A', 'H')
if dfs_path:
    print(f"Final DFS Path from A to H: {dfs_path}")
else:
    print("DFS: No path found.")

print("\n--- Let's try DFS for a different goal (I) ---")
dfs_path_i = dfs(graph, 'A', 'I')
if dfs_path_i:
    print(f"Final DFS Path from A to I: {dfs_path_i}")


--- Testing DFS ---

--- Running DFS from A to H ---
  Exploring node: A, current path: ['A']
  Exploring node: B, current path: ['A', 'B']
  Exploring node: D, current path: ['A', 'B', 'D']
  Exploring node: H, current path: ['A', 'B', 'D', 'H']
  Goal H reached!
Final DFS Path from A to H: ['A', 'B', 'D', 'H']

--- Let's try DFS for a different goal (I) ---

--- Running DFS from A to I ---
  Exploring node: A, current path: ['A']
  Exploring node: B, current path: ['A', 'B']
  Exploring node: D, current path: ['A', 'B', 'D']
  Exploring node: H, current path: ['A', 'B', 'D', 'H']
  Exploring node: J, current path: ['A', 'B', 'D', 'H', 'J']
  Exploring node: E, current path: ['A', 'B', 'E']
  Exploring node: C, current path: ['A', 'C']
  Exploring node: F, current path: ['A', 'C', 'F']
  Exploring node: I, current path: ['A', 'C', 'F', 'I']
  Goal I reached!
Final DFS Path from A to I: ['A', 'C', 'F', 'I']


In [ ]:
# --- UCS Example Run ---
print("\n--- Testing UCS ---")
ucs_path, ucs_cost = ucs(graph, 'A', 'H')
if ucs_path:
    print(f"Final UCS Path from A to H: {ucs_path}, Cost: {ucs_cost}")
else:
    print("UCS: No path found.")

print("\n--- Let's try UCS for the same goal (I) ---")
ucs_path_i, ucs_cost_i = ucs(graph, 'A', 'I')
if ucs_path_i:
    print(f"Final UCS Path from A to I: {ucs_path_i}, Cost: {ucs_cost_i}")


--- Testing UCS ---

--- Running UCS from A to H ---
  Exploring node: A (Cost: 0), current path: ['A']
  Exploring node: B (Cost: 1), current path: ['A', 'B']
  Exploring node: D (Cost: 4), current path: ['A', 'B', 'D']
  Exploring node: C (Cost: 5), current path: ['A', 'C']
  Exploring node: H (Cost: 5), current path: ['A', 'B', 'D', 'H']
  Goal H reached with total cost: 5!
Final UCS Path from A to H: ['A', 'B', 'D', 'H'], Cost: 5

--- Let's try UCS for the same goal (I) ---

--- Running UCS from A to I ---
  Exploring node: A (Cost: 0), current path: ['A']
  Exploring node: B (Cost: 1), current path: ['A', 'B']
  Exploring node: D (Cost: 4), current path: ['A', 'B', 'D']
  Exploring node: C (Cost: 5), current path: ['A', 'C']
  Exploring node: H (Cost: 5), current path: ['A', 'B', 'D', 'H']
  Exploring node: J (Cost: 6), current path: ['A', 'B', 'D', 'H', 'J']
  Exploring node: E (Cost: 7), current path: ['A', 'B', 'E']
  Exploring node: F (Cost: 7), current path: ['A', 'C', 'F']


**Note:** As you run the code, you might noticed how DFS dives deep into one branch before exploring others. It might find a path quickly, but not necessarily the shortest or cheapest one. UCS, on the other hand, prioritises paths with the lowest actual cost from the start, guaranteeing it finds the least-cost path. Pay attention to the ***Cost*** printed for each node in UCS, as it always expands the node with the lowest cumulative cost.

**B. Greedy Best-First Search vs. A* Search (as Best-First)*

Now let's move to informed search algorithms, which use a heuristic (an estimated guess) to guide their search. Greedy Best-First Search relies only on this guess, while A* Search (a common "***Best-First***" algorithm) smartly combines the actual cost so far with the heuristic.

In [ ]:
import heapq # Still for priority queue

# Define our graph again (same as before)
graph_heuristic = {
    'A': {'B': 1, 'C': 5},
    'B': {'D': 3, 'E': 6},
    'C': {'F': 2, 'G': 4},
    'D': {'H': 1},
    'E': {'H': 7},
    'F': {'I': 3},
    'G': {'I': 2},
    'H': {'J': 1},
    'I': {'J': 4},
    'J': {}
}

# Define a heuristic function h(n) for a goal 'J'
# This is an an informed "guess" of the cost from node n to the goal 'J'
# For A* to be optimal, this heuristic should be "admissible" (never overestimates true cost)
heuristic_j = {
    'A': 10, 'B': 9, 'C': 6, 'D': 8, 'E': 7,
    'F': 4, 'G': 3, 'H': 2, 'I': 1, 'J': 0
}

print("\n--- Graph for Greedy and A* ---")
for node, neighbors in graph_heuristic.items():
    print(f"{node}: {neighbors}")
print(f"Heuristic to J: {heuristic_j}")
print("-" * 30)



--- Graph for Greedy and A* ---
A: {'B': 1, 'C': 5}
B: {'D': 3, 'E': 6}
C: {'F': 2, 'G': 4}
D: {'H': 1}
E: {'H': 7}
F: {'I': 3}
G: {'I': 2}
H: {'J': 1}
I: {'J': 4}
J: {}
Heuristic to J: {'A': 10, 'B': 9, 'C': 6, 'D': 8, 'E': 7, 'F': 4, 'G': 3, 'H': 2, 'I': 1, 'J': 0}
------------------------------


In [ ]:
# --- Greedy Best-First Search ---
# Greedy Best-First always picks the node that LOOKS closest to the goal based on the heuristic.
# It doesn't consider the actual cost incurred so far.
def greedy_best_first_search(graph, start, goal, heuristic):
    """
    Performs Greedy Best-First Search.
    Uses heuristic only (h(n)) to prioritize nodes.
    Returns the path and the heuristic value at the goal.
    """
    # Priority queue stores (h_cost, current_node, path)
    priority_queue = [(heuristic[start], start, [start])]
    visited = set()

    print(f"\n--- Running Greedy Best-First Search from {start} to {goal} ---")

    while priority_queue:
        h_cost, current_node, path = heapq.heappop(priority_queue)
        print(f"  Exploring node: {current_node} (Heuristic: {h_cost}), current path: {path}")

        if current_node == goal:
            print(f"  Goal {goal} reached! Path: {path}")
            return path, h_cost # Note: h_cost here is just the last heuristic value

        if current_node in visited:
            continue
        visited.add(current_node)

        for neighbor in graph[current_node]:
            if neighbor not in visited:
                heapq.heappush(priority_queue, (heuristic[neighbor], neighbor, path + [neighbor]))
    print(f"  Goal {goal} not found.")
    return None, None


In [ ]:
# --- Greedy Best-First Search Example Run ---
print("\n--- Testing Greedy Best-First Search ---")
gbfs_path, _ = greedy_best_first_search(graph_heuristic, 'A', 'J', heuristic_j)
if gbfs_path:
    print(f"Final Greedy Best-First Path from A to J: {gbfs_path}")
else:
    print("Greedy Best-First: No path found.")


--- Testing Greedy Best-First Search ---

--- Running Greedy Best-First Search from A to J ---
  Exploring node: A (Heuristic: 10), current path: ['A']
  Exploring node: C (Heuristic: 6), current path: ['A', 'C']
  Exploring node: G (Heuristic: 3), current path: ['A', 'C', 'G']
  Exploring node: I (Heuristic: 1), current path: ['A', 'C', 'G', 'I']
  Exploring node: J (Heuristic: 0), current path: ['A', 'C', 'G', 'I', 'J']
  Goal J reached! Path: ['A', 'C', 'G', 'I', 'J']
Final Greedy Best-First Path from A to J: ['A', 'C', 'G', 'I', 'J']


In [ ]:
# --- A* Search ---
# A* search combines the actual cost from the start (g(n)) with the estimated cost to the goal (h(n)).
# It prioritizes nodes based on f(n) = g(n) + h(n), aiming for an optimal path.
def a_star_search(graph, start, goal, heuristic):
    """
    Performs A* Search.
    Uses f(n) = g(n) + h(n) to prioritize nodes.
    Returns the optimal path and total cost.
    """
    # Priority queue stores (f_cost, g_cost, current_node, path)
    priority_queue = [(heuristic[start], 0, start, [start])] # f_cost, g_cost, node, path
    visited_g_costs = {start: 0} # To store the lowest g_cost found to a node

    print(f"\n--- Running A* Search from {start} to {goal} ---")

    while priority_queue:
        f_cost, g_cost, current_node, path = heapq.heappop(priority_queue)
        print(f"  Exploring node: {current_node} (f_cost: {f_cost}, g_cost: {g_cost}), current path: {path}")

        # If we've found a cheaper path (lower g_cost) to this node previously, skip this one
        if g_cost > visited_g_costs.get(current_node, float('inf')):
            continue

        if current_node == goal:
            print(f"  Goal {goal} reached with total cost: {g_cost}!")
            return path, g_cost

        for neighbor, edge_cost in graph[current_node].items():
            new_g_cost = g_cost + edge_cost
            # Only add to queue if this new path to neighbor is cheaper than any previous one
            if new_g_cost < visited_g_costs.get(neighbor, float('inf')):
                visited_g_costs[neighbor] = new_g_cost
                new_f_cost = new_g_cost + heuristic[neighbor]
                heapq.heappush(priority_queue, (new_f_cost, new_g_cost, neighbor, path + [neighbor]))
    print(f"  Goal {goal} not found.")
    return None, None

In [ ]:
# --- A* Search Example Run ---
print("\n--- Testing A* Search ---")
a_star_path, a_star_cost = a_star_search(graph_heuristic, 'A', 'J', heuristic_j)
if a_star_path:
    print(f"Final A* Path from A to J: {a_star_path}, Cost: {a_star_cost}")
else:
    print("A*: No path found.")


--- Testing A* Search ---

--- Running A* Search from A to J ---
  Exploring node: A (f_cost: 10, g_cost: 0), current path: ['A']
  Exploring node: B (f_cost: 10, g_cost: 1), current path: ['A', 'B']
  Exploring node: C (f_cost: 11, g_cost: 5), current path: ['A', 'C']
  Exploring node: F (f_cost: 11, g_cost: 7), current path: ['A', 'C', 'F']
  Exploring node: I (f_cost: 11, g_cost: 10), current path: ['A', 'C', 'F', 'I']
  Exploring node: D (f_cost: 12, g_cost: 4), current path: ['A', 'B', 'D']
  Exploring node: H (f_cost: 7, g_cost: 5), current path: ['A', 'B', 'D', 'H']
  Exploring node: J (f_cost: 6, g_cost: 6), current path: ['A', 'B', 'D', 'H', 'J']
  Goal J reached with total cost: 6!
Final A* Path from A to J: ['A', 'B', 'D', 'H', 'J'], Cost: 6


**Note:** When you run above code, observe how Greedy Best-First Search might take what looks like the quickest path initially (based on `h(n)`), but it can sometimes get stuck or find a sub-optimal path. A* Search, by balancing `g(n)` (actual cost so far) and `h(n)` (estimated future cost), explores more intelligently to guarantee finding the optimal path (the one with the lowest total cost).

# **Part 2-2** Simulated Annealing & Local Beam Search
Now we move to local search algorithms, which are great for optimisation problems where we are trying to find the best possible configuration or value in a complex search space. We will use a simple mathematical function and try to find its minimum value.

In [ ]:
import random
import math

# Define a simple function to optimize (e.g., find minimum of x^2 - 4x + 5)
# The true minimum is at x=2, where f(x) = 1
def objective_function(x):
    return x**2 - 4*x + 5

# Define the search space bounds
MIN_X, MAX_X = -10, 10

# --- Simulated Annealing ---
# Simulated Annealing mimics the cooling of metals. It can accept "worse" moves sometimes
# (especially early on) to escape local minimums, then gradually settles into a good solution.
def simulated_annealing(objective_func, initial_temp, final_temp, alpha, max_iterations):
    """
    Performs Simulated Annealing to find the minimum of an objective function.
    """
    current_solution = random.uniform(MIN_X, MAX_X)
    current_energy = objective_func(current_solution)
    best_solution = current_solution
    best_energy = current_energy
    temperature = initial_temp

    print(f"\n--- Running Simulated Annealing ---")
    print(f"Initial solution: x={current_solution:.2f}, energy={current_energy:.2f}")

    for i in range(max_iterations):
        if temperature < final_temp:
            break

        # Generate a neighboring solution (e.g., small random step)
        # Step size often scales with temperature (larger steps at higher temps)
        step_size = (MAX_X - MIN_X) * 0.1 * (temperature / initial_temp)
        new_solution = current_solution + random.uniform(-step_size, step_size)

        # Keep new_solution within bounds
        new_solution = max(MIN_X, min(new_solution, MAX_X))

        new_energy = objective_func(new_solution)

        # Calculate energy difference (positive means worse/higher energy)
        delta_energy = new_energy - current_energy

        # If new solution is better (lower energy), accept it
        if delta_energy < 0:
            current_solution = new_solution
            current_energy = new_energy
            if new_energy < best_energy: # Update overall best found
                best_energy = new_energy
                best_solution = new_solution
        # If new solution is worse (higher energy), accept it with a certain probability
        else:
            acceptance_probability = math.exp(-delta_energy / temperature)
            if random.random() < acceptance_probability:
                current_solution = new_solution
                current_energy = new_energy

        # Cool down the temperature (e.g., multiply by a factor slightly less than 1)
        temperature *= alpha

        if i % (max_iterations // 10 or 1) == 0: # Print progress periodically
            print(f"Iteration {i}/{max_iterations}, Temp: {temperature:.2f}, Current: x={current_solution:.2f}, E={current_energy:.2f}, Best: x={best_solution:.2f}, E={best_energy:.2f}")

    print(f"\nSimulated Annealing Finished. Best found: x={best_solution:.2f}, Energy={best_energy:.2f}")
    return best_solution, best_energy


In [ ]:
# --- Simulated Annealing Example Run ---
print("\n--- Testing Simulated Annealing ---")
sa_solution, sa_energy = simulated_annealing(objective_function,
                                               initial_temp=100, # Starting temperature
                                               final_temp=0.1,   # Minimum temperature to stop
                                               alpha=0.99,       # Cooling rate (0.99 means temp reduces by 1% each iter)
                                               max_iterations=1000) # How many steps to take


--- Testing Simulated Annealing ---

--- Running Simulated Annealing ---
Initial solution: x=-0.17, energy=5.73
Iteration 0/1000, Temp: 99.00, Current: x=1.16, E=1.70, Best: x=1.16, E=1.70
Iteration 100/1000, Temp: 36.24, Current: x=2.97, E=1.95, Best: x=1.90, E=1.01
Iteration 200/1000, Temp: 13.26, Current: x=0.03, E=4.90, Best: x=2.02, E=1.00
Iteration 300/1000, Temp: 4.86, Current: x=-0.69, E=8.21, Best: x=2.02, E=1.00
Iteration 400/1000, Temp: 1.78, Current: x=-0.34, E=6.49, Best: x=2.02, E=1.00
Iteration 500/1000, Temp: 0.65, Current: x=-0.22, E=5.93, Best: x=2.02, E=1.00
Iteration 600/1000, Temp: 0.24, Current: x=-0.11, E=5.46, Best: x=2.02, E=1.00

Simulated Annealing Finished. Best found: x=2.02, Energy=1.00


In [ ]:
# --- Local Beam Search ---
# Local Beam Search maintains multiple candidate solutions (k "beams") at once.
# In each step, it explores neighbors from ALL beams and picks the k BEST overall neighbors for the next step.
def local_beam_search(objective_func, k, max_iterations):
    """
    Performs Local Beam Search to find the minimum of an objective function.
    """
    # Initialise k random solutions (beams)
    current_solutions = [random.uniform(MIN_X, MAX_X) for _ in range(k)]
    current_energies = [objective_func(x) for x in current_solutions]

    print(f"\n--- Running Local Beam Search (k={k}) ---")
    print(f"Initial solutions (k beams): {[f'x={x:.2f}, E={e:.2f}' for x, e in zip(current_solutions, current_energies)]}")

    for i in range(max_iterations):
        next_candidates = []
        for sol_idx, sol in enumerate(current_solutions):
            # Generate neighbors for each current solution (e.g., 5 random neighbors)
            for _ in range(5):
                step_size = (MAX_X - MIN_X) * 0.1 # Example constant step size
                neighbor = sol + random.uniform(-step_size, step_size)
                neighbor = max(MIN_X, min(neighbor, MAX_X)) # Keep within bounds
                next_candidates.append((objective_func(neighbor), neighbor))

        # Select the k best candidates from ALL generated neighbors
        next_candidates.sort(key=lambda x: x[0]) # Sort by energy (lower is better)

        current_solutions = [x[1] for x in next_candidates[:k]]
        current_energies = [x[0] for x in next_candidates[:k]]

        if i % (max_iterations // 10 or 1) == 0: # Print progress periodically
            print(f"Iteration {i}/{max_iterations}, Best k solutions: {[f'x={x:.2f}, E={e:.2f}' for x, e in zip(current_solutions, current_energies)]}")

    best_solution_lbs = min(current_solutions, key=objective_func)
    best_energy_lbs = objective_function(best_solution_lbs)
    print(f"\nLocal Beam Search Finished. Best found: x={best_solution_lbs:.2f}, Energy={best_energy_lbs:.2f}")
    return best_solution_lbs, best_energy_lbs


In [ ]:
# --- Local Beam Search Example Run ---
print("\n--- Testing Local Beam Search ---")
lbs_solution, lbs_energy = local_beam_search(objective_function,
                                               k=5, # Number of parallel beams/solutions to maintain
                                               max_iterations=200) # How many iterations


--- Testing Local Beam Search ---

--- Running Local Beam Search (k=5) ---
Initial solutions (k beams): ['x=-4.39, E=41.85', 'x=-7.20, E=85.71', 'x=-3.70, E=33.52', 'x=1.12, E=1.77', 'x=3.45, E=3.09']
Iteration 0/200, Best k solutions: ['x=1.96, E=1.00', 'x=2.49, E=1.24', 'x=2.60, E=1.35', 'x=2.60, E=1.36', 'x=1.33, E=1.45']
Iteration 20/200, Best k solutions: ['x=1.76, E=1.06', 'x=1.75, E=1.06', 'x=2.31, E=1.10', 'x=1.63, E=1.14', 'x=1.36, E=1.41']
Iteration 40/200, Best k solutions: ['x=1.99, E=1.00', 'x=2.08, E=1.01', 'x=1.81, E=1.04', 'x=2.21, E=1.05', 'x=1.74, E=1.07']
Iteration 60/200, Best k solutions: ['x=2.05, E=1.00', 'x=1.93, E=1.00', 'x=1.78, E=1.05', 'x=1.59, E=1.17', 'x=2.53, E=1.28']
Iteration 80/200, Best k solutions: ['x=1.88, E=1.01', 'x=2.18, E=1.03', 'x=1.77, E=1.05', 'x=1.72, E=1.08', 'x=2.32, E=1.10']
Iteration 100/200, Best k solutions: ['x=2.09, E=1.01', 'x=2.20, E=1.04', 'x=1.75, E=1.06', 'x=1.72, E=1.08', 'x=1.60, E=1.16']
Iteration 120/200, Best k solutions:

**Note:**

For **Simulated Annealing**, watch how the "***Temperature***" decreases over iterations. Early on, you might see the "***Current***" energy occasionally get worse (higher), which is the algorithm deliberately exploring to escape local traps. As temperature drops, it becomes less likely to accept worse moves and focuses on refining the best solution.

For **Local Beam Search**, you'll see a list of `k` solutions for each iteration. Notice how these `k` solutions evolve. The algorithm always picks the best `k` solutions from all explored neighbours, allowing it to explore promising areas in parallel and potentially converge faster or avoid some local optima.

# **Part 2-3** Genetic Algorithms
Finally, we will explore Genetic Algorithms (GAs), which are inspired by biological evolution. They "***evolve***" a population of candidate solutions over generations using concepts like selection, crossover, and mutation to find good solutions to complex problems. We will try to evolve a random bitstring to match a target string.

In [ ]:
import random

# --- Genetic Algorithm ---
# Genetic Algorithms simulate natural selection to evolve solutions.
# They maintain a 'population' of solutions, select the 'fittest',
# and then 'reproduce' them with 'crossover' and 'mutation' to create new generations.
def genetic_algorithm(target_string, population_size, mutation_rate, generations):
    """
    Performs a Genetic Algorithm to evolve a bitstring to match a target string.
    """
    target_length = len(target_string)

    # 1. Initialize Population: Create a set of random initial solutions
    population = []
    for _ in range(population_size):
        individual = ''.join(random.choice(['0', '1']) for _ in range(target_length))
        population.append(individual)

    print(f"\n--- Running Genetic Algorithm ---")
    print(f"Target String: {target_string}")
    print(f"Initial population (first 5 samples): {population[:min(5, population_size)]}...")

    for gen in range(generations):
        # 2. Evaluate Fitness: Determine how 'good' each solution is
        # Here, fitness is the number of characters that match the target string
        fitness_scores = []
        for individual in population:
            score = sum(1 for i in range(target_length) if individual[i] == target_string[i])
            fitness_scores.append(score)

        # Keep track of the best individual in this generation
        best_individual_idx = fitness_scores.index(max(fitness_scores))
        best_individual = population[best_individual_idx]
        best_fitness = fitness_scores[best_individual_idx]

        # Check if we've reached the goal
        if best_fitness == target_length:
            print(f"\nGoal reached at Generation {gen}: '{best_individual}' (Fitness: {best_fitness})")
            return best_individual, gen

        # Print progress periodically
        if gen % (generations // 10 or 1) == 0:
            print(f"Generation {gen}: Best individual = '{best_individual}' (Fitness: {best_fitness}/{target_length})")

        # 3. Selection: Choose individuals for reproduction based on their fitness
        # Higher fitness means higher chance of being selected (Roulette Wheel style)
        total_fitness = sum(fitness_scores)
        if total_fitness == 0: # Handle case where all fitnesses are zero to avoid division by zero
            selection_probabilities = [1 / population_size] * population_size
        else:
            selection_probabilities = [f / total_fitness for f in fitness_scores]

        # Create the next generation
        new_population = []
        for _ in range(population_size // 2): # Create pairs of offspring until new population is full
            # Select two parents based on fitness probabilities
            parent1 = random.choices(population, weights=selection_probabilities, k=1)[0]
            parent2 = random.choices(population, weights=selection_probabilities, k=1)[0]

            # 4. Crossover (Recombination): Combine genetic material from two parents
            # Here, we use single-point crossover: parents swap parts of their strings
            crossover_point = random.randint(1, target_length - 1)
            child1 = parent1[:crossover_point] + parent2[crossover_point:]
            child2 = parent2[:crossover_point] + parent1[crossover_point:]

            # 5. Mutation: Introduce small, random changes to maintain diversity
            def mutate(individual, rate):
                mutated_individual = list(individual)
                for i in range(target_length):
                    if random.random() < rate: # With a small probability, flip a bit
                        mutated_individual[i] = '1' if mutated_individual[i] == '0' else '0'
                return "".join(mutated_individual)

            new_population.append(mutate(child1, mutation_rate))
            new_population.append(mutate(child2, mutation_rate))

        population = new_population # Replace old population with the new generation

    print(f"\nGenetic Algorithm Finished. Best found: '{best_individual}' (Fitness: {best_fitness}/{target_length})")
    return best_individual, best_fitness

# --- Genetic Algorithm Example Run ---
print("\n--- Testing Genetic Algorithm ---")
target_dna = "1011010011010101" # Our target sequence of 0s and 1s
ga_best_solution, ga_generations = genetic_algorithm(target_dna,
                                                    population_size=100,   # Number of individuals in each generation
                                                    mutation_rate=0.01,    # Probability of a bit flipping (e.0. 1%)
                                                    generations=500)       # How many generations to run


--- Testing Genetic Algorithm ---

--- Running Genetic Algorithm ---
Target String: 1011010011010101
Initial population (first 5 samples): ['1000101111011001', '0011101101011000', '0101110110100100', '0110001011110000', '0101101101110100']...
Generation 0: Best individual = '1011010011000001' (Fitness: 14/16)

Goal reached at Generation 12: '1011010011010101' (Fitness: 16)


**Note:**
As you run the Genetic Algorithm, watch how the "***Best individual***" for each generation gets progressively closer to the `Target String`. The "***Fitness***" score should generally increase over time. This illustrates how, through simulated evolution – where the fittest individuals survive and reproduce, and random mutations introduce new possibilities – the algorithm can effectively search for and find optimal or near-optimal solutions to complex problems.